# Instalación de Spark

In [1]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://archive.apache.org/dist/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

import os
import sys

import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark= SparkSession.builder.appName("Mi primera").getOrCreate()

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,190 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,523 kB]
Get:13 http://archive.ubuntu.com/ubuntu 

## Clonación del Repositorio de GitHub

Se hace clonación del repositorio de GitHub que contiene las imágenes.
[Fruits-360](https://github.com/fruits-360/fruits-360-100x100)



In [2]:
!apt-get install git

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.11).
0 upgraded, 0 newly installed, 0 to remove and 50 not upgraded.


In [3]:
repository_url = "https://github.com/fruits-360/fruits-360-100x100"  # Replace with your repo URL
!git clone {repository_url}

Cloning into 'fruits-360-100x100'...
remote: Enumerating objects: 94555, done.
remote: Counting objects: 100% (3809/3809), done.
remote: Compressing objects: 100% (3799/3799), done.
remote: Total 94555 (delta 32), reused 3786 (delta 10), pack-reused 90746 (from 1)
Receiving objects: 100% (94555/94555), 723.85 MiB | 32.04 MiB/s, done.
Resolving deltas: 100% (33/33), done.
Updating files: 100% (94112/94112), done.


Se genera un dataframe con las características principales de todas las imágenes dentro de la carpeta Training.

In [4]:
import os
import cv2
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Image Metadata DataFrame with Subfolders") \
    .getOrCreate()

# Folder containing the images
image_folder = "/content/fruits-360-100x100/Training"

# Function to calculate average color and colorfulness index
def calculate_color_features(image):
    # Convert image to RGB if necessary
    if image.shape[-1] == 3:
        # Split into channels
        b, g, r = cv2.split(image)
        # Average color
        avg_color = [float(np.mean(r)), float(np.mean(g)), float(np.mean(b))]
        # Colorfulness Index
        rg = np.abs(r - g)
        yb = np.abs(0.5 * (r + g) - b)
        colorfulness = float(np.mean(rg) + np.mean(yb))
        return avg_color, colorfulness
    else:
        return [0.0, 0.0, 0.0], 0.0  # Defaults for grayscale images

# Function to calculate statistical features
def calculate_statistical_features(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # Convert to grayscale
    mean = float(np.mean(gray))
    variance = float(np.var(gray))
    std_dev = float(np.std(gray))
    return mean, variance, std_dev

# Function to process each image
def process_image(image_path):
    try:
        # Read the image
        img = cv2.imread(image_path)
        if img is None:
            return None

        # Get file details
        file_name = os.path.basename(image_path)
        file_size = os.path.getsize(image_path)
        file_format = file_name.split(".")[-1].upper()
        dimensions = img.shape[:2]  # (height, width)

        # Calculate features
        avg_color, colorfulness = calculate_color_features(img)
        mean, variance, std_dev = calculate_statistical_features(img)

        # Return a dictionary of the features
        return {
            "file_name": file_name,
            "file_path": image_path,  # Include full path for subfolders
            "file_size": file_size,
            "file_format": file_format,
            "width": dimensions[1],
            "height": dimensions[0],
            "avg_color_r": avg_color[0],
            "avg_color_g": avg_color[1],
            "avg_color_b": avg_color[2],
            "colorfulness": colorfulness,
            "mean_intensity": mean,
            "variance_intensity": variance,
            "std_dev_intensity": std_dev
        }
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

# Traverse the image_folder and all its subfolders
data = []
for root, _, files in os.walk(image_folder):
    for filename in files:
        file_path = os.path.join(root, filename)
        features = process_image(file_path)
        if features:
            data.append(features)

# Define schema for PySpark DataFrame
schema = StructType([
    StructField("file_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("file_size", IntegerType(), True),
    StructField("file_format", StringType(), True),
    StructField("width", IntegerType(), True),
    StructField("height", IntegerType(), True),
    StructField("avg_color_r", FloatType(), True),
    StructField("avg_color_g", FloatType(), True),
    StructField("avg_color_b", FloatType(), True),
    StructField("colorfulness", FloatType(), True),
    StructField("mean_intensity", FloatType(), True),
    StructField("variance_intensity", FloatType(), True),
    StructField("std_dev_intensity", FloatType(), True),
])

# Create PySpark DataFrame
dataframe = spark.createDataFrame(data, schema)

# Show the resulting DataFrame
dataframe.show(truncate=False)


+--------------+--------------------------------------------------------------+---------+-----------+-----+------+-----------+-----------+-----------+------------+--------------+------------------+-----------------+
|file_name     |file_path                                                     |file_size|file_format|width|height|avg_color_r|avg_color_g|avg_color_b|colorfulness|mean_intensity|variance_intensity|std_dev_intensity|
+--------------+--------------------------------------------------------------+---------+-----------+-----+------+-----------+-----------+-----------+------------+--------------+------------------+-----------------+
|r0_213_100.jpg|/content/fruits-360-100x100/Training/Zucchini 1/r0_213_100.jpg|1198     |JPG        |100  |100   |230.4415   |232.4195   |221.5112   |177.8482    |230.5838      |2323.6436         |48.204185        |
|r0_234_100.jpg|/content/fruits-360-100x100/Training/Zucchini 1/r0_234_100.jpg|1195     |JPG        |100  |100   |230.3321   |232.3423  

Agregamos la columna category que contenga el nombre de la fruta/verdura de la imagen.

In [5]:
from pyspark.sql.functions import split

# Add a new column extracting the desired part of the file_path
dataframe = dataframe.withColumn(
    "category",
    split(dataframe["file_path"], "/").getItem(4)  # Extract the 4th index (5th segment)
)

# Show the updated DataFrame
dataframe.show(truncate=False)


+--------------+--------------------------------------------------------------+---------+-----------+-----+------+-----------+-----------+-----------+------------+--------------+------------------+-----------------+----------+
|file_name     |file_path                                                     |file_size|file_format|width|height|avg_color_r|avg_color_g|avg_color_b|colorfulness|mean_intensity|variance_intensity|std_dev_intensity|category  |
+--------------+--------------------------------------------------------------+---------+-----------+-----+------+-----------+-----------+-----------+------------+--------------+------------------+-----------------+----------+
|r0_213_100.jpg|/content/fruits-360-100x100/Training/Zucchini 1/r0_213_100.jpg|1198     |JPG        |100  |100   |230.4415   |232.4195   |221.5112   |177.8482    |230.5838      |2323.6436         |48.204185        |Zucchini 1|
|r0_234_100.jpg|/content/fruits-360-100x100/Training/Zucchini 1/r0_234_100.jpg|1195     |JPG

Realizar análisis con MLlib de pyspark a tu conjunto de datos.

Ajustamos el dataframe para que se le pueda aplicar logistic regression.

In [15]:
from pyspark.ml.feature import VectorAssembler

inputCols = [
    'file_size',
    'avg_color_r',
    'avg_color_g',
    'avg_color_b',
    'colorfulness',
    'mean_intensity',
    'variance_intensity',
    'std_dev_intensity'
]
assembler = VectorAssembler(inputCols = inputCols, outputCol = "features")
df_with_features = assembler.transform(dataframe)

df_with_features.show(3)

+--------------+--------------------+---------+-----------+-----+------+-----------+-----------+-----------+------------+--------------+------------------+-----------------+----------+--------------------+
|     file_name|           file_path|file_size|file_format|width|height|avg_color_r|avg_color_g|avg_color_b|colorfulness|mean_intensity|variance_intensity|std_dev_intensity|  category|            features|
+--------------+--------------------+---------+-----------+-----+------+-----------+-----------+-----------+------------+--------------+------------------+-----------------+----------+--------------------+
|r0_213_100.jpg|/content/fruits-3...|     1198|        JPG|  100|   100|   230.4415|   232.4195|   221.5112|    177.8482|      230.5838|         2323.6436|        48.204185|Zucchini 1|[1198.0,230.44149...|
|r0_234_100.jpg|/content/fruits-3...|     1195|        JPG|  100|   100|   230.3321|   232.3423|   221.6652|    181.8552|       230.517|          2293.282|         47.88822|Zuc

In [19]:
# Selección de características usando Chi-Square
from pyspark.ml.feature import ChiSqSelector, StringIndexer


indexer = StringIndexer(inputCol = "category", outputCol = "category_index")
df_indexado = indexer.fit(df_with_features).transform(df_with_features)

df_indexado.show(3)

+--------------+--------------------+---------+-----------+-----+------+-----------+-----------+-----------+------------+--------------+------------------+-----------------+----------+--------------------+--------------+
|     file_name|           file_path|file_size|file_format|width|height|avg_color_r|avg_color_g|avg_color_b|colorfulness|mean_intensity|variance_intensity|std_dev_intensity|  category|            features|category_index|
+--------------+--------------------+---------+-----------+-----+------+-----------+-----------+-----------+------------+--------------+------------------+-----------------+----------+--------------------+--------------+
|r0_213_100.jpg|/content/fruits-3...|     1198|        JPG|  100|   100|   230.4415|   232.4195|   221.5112|    177.8482|      230.5838|         2323.6436|        48.204185|Zucchini 1|[1198.0,230.44149...|         135.0|
|r0_234_100.jpg|/content/fruits-3...|     1195|        JPG|  100|   100|   230.3321|   232.3423|   221.6652|    181.

In [22]:
# Split the data into training and testing sets
train_df, test_df = df_indexado.randomSplit([0.8, 0.2], seed=42)


In [24]:
# Logistic Regression for Classification
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Define and train the model
lr = LogisticRegression(featuresCol="features", labelCol="category_index")
lr_model = lr.fit(train_df)

# Evaluate the model
predictions = lr_model.transform(test_df)
evaluator = MulticlassClassificationEvaluator(labelCol="category_index", predictionCol="prediction",
                                              metricName="accuracy")
accuracy = evaluator.evaluate(predictions)


In [25]:
accuracy

0.9272298793115761